# 3강. 왜 단순 next-character 예측만으로는 의미 있는 임베딩이 부족한가?

2강에서 우리는 이런 모델을 만들었습니다.

```text id="bj9vds"
현재 문자
→ embedding
→ linear
→ 다음 문자 예측
```

이 모델도 embedding을 학습합니다. 하지만 여기서 조심해야 합니다.

> **학습된 임베딩 = 좋은 의미 임베딩**은 아닙니다.

임베딩은 항상 어떤 목적함수, 즉 **loss objective**에 맞게 학습됩니다. Word2Vec의 CBOW/Skip-gram은 단어의 주변 문맥을 예측하도록 설계되었고, GloVe는 word-word co-occurrence matrix의 통계를 활용합니다. 반면 2강의 toy model은 단지 “현재 문자 하나로 다음 문자 하나를 맞히는 문제”였습니다. 목적함수가 다르므로 임베딩 공간의 성질도 달라집니다. ([arXiv][1])

---

## 3-1. 2강 모델의 한계

2강 모델은 사실상 **bigram model**입니다.

```text id="9h118o"
P(next_token | current_token)
```

즉, 현재 token 하나만 보고 다음 token을 예측합니다.

예를 들어:

```text id="f8ovnn"
h → e
e → l
l → l
l → o
```

이런 식입니다.

문제는 이 모델이 문맥을 거의 보지 못한다는 점입니다.

```text id="k1gjxj"
"bank account"
"river bank"
```

여기서 `bank`는 문맥에 따라 의미가 다릅니다.

하지만 단순 bigram 모델이 현재 token만 본다면:

```text id="bavrl4"
bank → ?
```

정도만 학습합니다.
`bank` 앞에 `river`가 있었는지, `money`가 있었는지, `account`가 뒤에 있는지 충분히 활용하지 못합니다.

반면 BERT는 모든 layer에서 좌우 문맥을 함께 조건화하는 bidirectional representation을 학습하도록 설계되었습니다. 그래서 같은 단어라도 문맥에 따라 다른 hidden representation을 만들 수 있습니다. ([arXiv][2])

---

# 3-2. 왜 character-level은 의미 학습이 어렵나?

문자 단위 모델은 입력 단위가 너무 작습니다.

```text id="pn05k9"
"embedding"
→ ["e", "m", "b", "e", "d", "d", "i", "n", "g"]
```

여기서 `"e"` 자체는 의미 단위가 아닙니다.
`"m"`도 의미 단위가 아닙니다.
`"d"`도 의미 단위가 아닙니다.

물론 대규모 데이터와 충분한 모델을 쓰면 character-level 모델도 패턴을 학습할 수 있습니다. 하지만 우리가 만든 작은 모델에서는 문자 하나하나가 의미 있는 단위가 아니기 때문에 semantic embedding을 기대하기 어렵습니다.

단어 임베딩에서는 보통 이런 단위가 더 중요합니다.

```text id="af4klv"
machine
learning
neural
network
embedding
retrieval
```

이런 단어들은 주변 문맥을 통해 의미적 관계를 학습할 수 있습니다.

---

# 3-3. “학습된다”와 “의미 있다”의 차이

2강 모델에서도 embedding vector는 바뀝니다.

```text id="f7spjb"
random embedding
→ loss 계산
→ gradient
→ optimizer update
→ embedding 변화
```

그런데 그 변화는 다음 목적에 맞춰져 있습니다.

```text id="i4qv2m"
현재 문자를 보고 다음 문자를 맞히기
```

따라서 학습된 embedding은 이런 정보에는 민감할 수 있습니다.

```text id="7xkys0"
h 다음에는 e가 자주 온다
q 다음에는 u가 자주 온다
공백 다음에는 새 단어가 온다
```

하지만 이런 의미 관계를 잘 학습한다고 보기는 어렵습니다.

```text id="7fg2ev"
cat ≈ dog
king ≈ queen
machine learning ≈ deep learning
retrieval ≈ search
```

즉, 2강 모델이 배운 것은 주로 **문자 전이 패턴**입니다.
우리가 원하는 것은 **문맥 기반 의미 구조**입니다.

---

# 3-4. 좋은 임베딩에는 무엇이 필요한가?

의미 있는 임베딩을 만들려면 최소한 다음이 필요합니다.

| 필요 요소         | 이유                                        |
| ------------- | ----------------------------------------- |
| 적절한 token 단위  | 문자보다 단어/subword가 의미 학습에 유리                |
| 충분한 context   | 단어 하나만 보지 말고 주변 단어를 봐야 함                  |
| 적절한 objective | 의미적으로 가까운 항목이 가까워지도록 학습해야 함               |
| 충분한 데이터       | 작은 corpus에서는 우연한 패턴에 쉽게 과적합               |
| 평가 기준         | loss만 보지 말고 similarity/retrieval 품질을 봐야 함 |

Word2Vec은 이 중에서 **context prediction**을 핵심으로 씁니다. Skip-gram은 중심 단어로 주변 단어를 예측하고, CBOW는 주변 단어로 중심 단어를 예측하는 방식입니다. ([arXiv][1])

---

# 3-5. next-character prediction vs context-based embedding

차이를 표로 보면 명확합니다.

| 구분            | 2강 toy model     | Word2Vec Skip-gram     |
| ------------- | ---------------- | ---------------------- |
| 입력 단위         | 문자               | 단어                     |
| 입력 정보         | 현재 문자 1개         | 중심 단어                  |
| 예측 대상         | 다음 문자            | 주변 단어들                 |
| 학습 신호         | local transition | distributional context |
| 임베딩 성질        | 다음 문자 예측에 유리     | 문맥이 비슷한 단어가 가까워짐       |
| 의미 임베딩으로 적합한가 | 약함               | 훨씬 적합                  |

GloVe도 같은 방향의 문제의식을 갖습니다. 다만 neural context prediction 대신, 전체 corpus에서 단어들이 얼마나 함께 등장하는지 세는 **global word-word co-occurrence matrix**를 사용합니다. ([스탠포드 자연어처리 그룹][3])

---

# 3-6. 작은 예제로 직관 보기

다음 corpus를 봅시다.

In [1]:
corpus = [
    "cat eats fish",
    "dog eats meat",
    "cat likes milk",
    "dog likes bone",
    "car uses fuel",
    "truck uses diesel",
    "car has wheels",
    "truck has wheels",
]

사람 입장에서는 다음 관계가 보입니다.

```text id="mslss3"
cat ↔ dog     # 동물
car ↔ truck   # 차량
fish ↔ meat   # 먹이
fuel ↔ diesel # 연료
```

그런데 단순 next-word 모델은 이런 식으로 봅니다.

```text id="p1szxp"
cat → eats
dog → eats
cat → likes
dog → likes
car → uses
truck → uses
car → has
truck → has
```

이 경우 `cat`과 `dog`는 둘 다 `eats`, `likes` 같은 비슷한 다음 단어를 가집니다.
`car`와 `truck`도 둘 다 `uses`, `has` 같은 비슷한 다음 단어를 가집니다.

그래서 word-level next-token prediction은 character-level보다는 낫습니다.

하지만 여전히 한계가 있습니다.

```text id="5ybcwk"
현재 단어 하나만 보고 다음 단어만 맞힌다.
앞뒤 전체 문맥을 넓게 보지 않는다.
문장 전체 의미를 직접 학습하지 않는다.
```

그래서 다음 단계에서는 **주변 window**를 사용합니다.

```text id="qro4qn"
center word 주변 ±2개 단어를 context로 본다.
```

예를 들어:

```text id="wvvyxx"
"deep learning uses neural networks"
```

window size가 2이고 center word가 `uses`라면:

```text id="vo7zgi"
center: uses
context: deep, learning, neural, networks
```

이런 식입니다.

---

# 3-7. context pair 직접 만들어보기

오늘의 실습은 학습까지 가지 않고, **Word2Vec으로 넘어가기 위한 데이터 구조**만 만듭니다.

In [2]:
from collections import Counter

corpus = [
    "cat eats fish",
    "dog eats meat",
    "cat likes milk",
    "dog likes bone",
    "car uses fuel",
    "truck uses diesel",
    "car has wheels",
    "truck has wheels",
]

tokenized = [sent.split() for sent in corpus]

vocab = sorted(set(word for sent in tokenized for word in sent))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

print("vocab:", vocab)

vocab: ['bone', 'car', 'cat', 'diesel', 'dog', 'eats', 'fish', 'fuel', 'has', 'likes', 'meat', 'milk', 'truck', 'uses', 'wheels']


이제 Skip-gram용 `(center, context)` pair를 만듭니다.

In [3]:
def make_skipgram_pairs(tokenized, window_size=2):
    pairs = []

    for sent in tokenized:
        for center_pos, center_word in enumerate(sent):
            left = max(0, center_pos - window_size)
            right = min(len(sent), center_pos + window_size + 1)

            for context_pos in range(left, right):
                if context_pos == center_pos:
                    continue

                context_word = sent[context_pos]
                pairs.append((center_word, context_word))

    return pairs

pairs = make_skipgram_pairs(tokenized, window_size=2)

for p in pairs[:20]:
    print(p)

print("num pairs:", len(pairs))

('cat', 'eats')
('cat', 'fish')
('eats', 'cat')
('eats', 'fish')
('fish', 'cat')
('fish', 'eats')
('dog', 'eats')
('dog', 'meat')
('eats', 'dog')
('eats', 'meat')
('meat', 'dog')
('meat', 'eats')
('cat', 'likes')
('cat', 'milk')
('likes', 'cat')
('likes', 'milk')
('milk', 'cat')
('milk', 'likes')
('dog', 'likes')
('dog', 'bone')
num pairs: 48


출력은 이런 형태입니다.

```text id="49fpoa"
('cat', 'eats')
('cat', 'fish')
('eats', 'cat')
('eats', 'fish')
('fish', 'cat')
('fish', 'eats')
...
```

이 pair들은 다음 모델을 학습시키는 데 사용됩니다.

```text id="s7qk3y"
center word → context word 예측
```

이게 바로 Skip-gram의 핵심입니다.

---

# 3-8. co-occurrence matrix도 만들어보기

Skip-gram은 neural objective로 context를 예측합니다.
그 전에 더 단순하게 co-occurrence matrix를 만들 수도 있습니다.

In [4]:
import numpy as np

V = len(vocab)
cooc = np.zeros((V, V), dtype=np.float32)

window_size = 2

for sent in tokenized:
    ids = [word2idx[w] for w in sent]

    for center_pos, center_id in enumerate(ids):
        left = max(0, center_pos - window_size)
        right = min(len(ids), center_pos + window_size + 1)

        for context_pos in range(left, right):
            if context_pos == center_pos:
                continue

            context_id = ids[context_pos]
            cooc[center_id, context_id] += 1

print("cooc shape:", cooc.shape)
print(cooc)

cooc shape: (15, 15)
[[0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 0. 1. 1. 0. 0. 1. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0.]
 [1. 0. 0. 0. 0. 1. 0. 0. 0. 1. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 1. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 2.]
 [1. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 1.]
 [0. 1. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 2. 0. 0. 0. 1. 0. 0.]]


여기서 row 하나가 한 단어의 context profile입니다.

```text id="k1b0v2"
cooc[word_id] = 해당 단어가 어떤 단어들과 같이 등장했는지 나타내는 벡터
```

이제 cosine similarity를 계산해볼 수 있습니다.

In [5]:
import numpy as np

def cosine(a, b, eps=1e-8):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps)

def nearest(word, matrix, top_k=5):
    wid = word2idx[word]
    query = matrix[wid]

    scores = []
    for i in range(matrix.shape[0]):
        if i == wid:
            continue
        scores.append((idx2word[i], cosine(query, matrix[i])))

    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

for word in ["cat", "dog", "car", "truck"]:
    print("\nQUERY:", word)
    print(nearest(word, cooc))


QUERY: cat
[('dog', np.float32(0.5)), ('bone', np.float32(0.35355338)), ('fish', np.float32(0.35355338)), ('meat', np.float32(0.35355338)), ('milk', np.float32(0.35355338))]

QUERY: dog
[('cat', np.float32(0.5)), ('bone', np.float32(0.35355338)), ('fish', np.float32(0.35355338)), ('meat', np.float32(0.35355338)), ('milk', np.float32(0.35355338))]

QUERY: car
[('truck', np.float32(0.75)), ('has', np.float32(0.40824828)), ('wheels', np.float32(0.40824828)), ('diesel', np.float32(0.35355338)), ('fuel', np.float32(0.35355338))]

QUERY: truck
[('car', np.float32(0.75)), ('has', np.float32(0.40824828)), ('wheels', np.float32(0.40824828)), ('diesel', np.float32(0.35355338)), ('fuel', np.float32(0.35355338))]


기대하는 관찰은 이겁니다.

```text id="qh42s4"
cat의 주변 문맥 ≈ dog의 주변 문맥
car의 주변 문맥 ≈ truck의 주변 문맥
```

데이터가 작아서 완벽하지는 않지만, 적어도 2강의 character-level next prediction보다 “의미 관계”에 가까운 학습 신호가 생깁니다.

---

# 3-9. 이번 강의의 핵심

오늘 핵심은 이겁니다.

```text id="f8n87q"
임베딩의 품질은 embedding layer 자체가 아니라,
무엇을 예측하도록 학습했는지에 의해 결정된다.
```

`nn.Embedding`은 그냥 lookup table입니다.

좋은 임베딩을 만들려면 좋은 학습 문제가 필요합니다.

```text id="otmxn8"
나쁜/약한 objective:
현재 문자 하나로 다음 문자 하나 맞히기

더 나은 objective:
중심 단어로 주변 단어 예측하기
주변 단어로 중심 단어 예측하기
비슷한 문장은 가깝게, 다른 문장은 멀게 만들기
검색 query와 relevant document를 가깝게 만들기
```

임베딩 모델 연구의 대부분은 결국 이 질문으로 귀결됩니다.

> **어떤 객체들을 가깝게 만들고, 어떤 객체들을 멀게 만들 것인가?**